In [73]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [74]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [ ]:
query_com = """
SELECT DISTINCT c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2025-07-19'
    AND v.Fecha <= '2025-07-19'+
    AND Canal IN ('Retail','WhatsApp')
    AND v.Venta_Neta > 0
GROUP BY  c.Cliente,
       c.Nombres + ' ' + c.Apellidos,
       c.Email,
       c.Celular,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,C1001369188,MARIA JOSE GUTIERREZ,majose0827@gmail.com,3124667861,2025-07-19,CHATCENTER WEB,297256.12
1,C1002155758,CATALINA ANAYA,cata.anaya.arevalo@gmail.com,3228724985,2025-07-19,CHATCENTER WEB,141170.40
2,C1010188841,PAULA ÁLVAREZ,paulis.422@gmail.com,3115493159,2025-07-19,CHATCENTER WEB,322675.20
3,C1010207508,LAURA MELISA VEGA CASTRO,lmelisavega@gmail.com,3045774668,2025-07-19,CHATCENTER WEB,268475.85
4,C1017196416,LUISA FERNANDA GOMEZ ARANGO,LUISAGOMEZAR@HOTMAIL.COM,3122668619,2025-07-19,Chat Center,158823.53


In [76]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-07-19'
#     AND v.Fecha <= '2025-07-19'
#     AND Canal IN ('CHATCENTER WEB','Chat Center')
#     AND v.Venta_Neta > 0
#     AND v.Linea = 'ALMENDRA'
# GROUP BY  c.Cliente,
#        c.Nombres + ' ' + c.Apellidos,
#        c.Email,
#        c.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)

# df_ventas.head()

In [77]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
# 	Canal,
#     SUM(Venta_Neta) AS Venta 
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE Codigo_Descuento LIKE '%CUMP%'
#     AND Canal IN ('Chat Center', 'CHATCENTER WEB')
# 	AND Fecha BETWEEN '2025-07-01' AND '2025-07-31'
# 	AND NOT EXISTS (SELECT 1 FROM COMERSSIA.PROVENZAL.dbo.EmpleadosProvenzal e WHERE e.Codigo = v.cliente ) 
# GROUP BY  v.Cliente,
#        cc.Nombres + ' ' +  cc.Apellidos,
#        cc.Email,
#        cc.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [78]:
# Leer archivo
df_social = pd.read_excel("Atribucion.xlsx")

df_social['contactid'] = df_social['contactid'].astype(str).str[2:]

In [79]:
# Normalizar df_ventas
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_atribucion = pd.merge(df_ventas, df_social, left_on='Celular', right_on='contactid' , how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.0f}")
# Resultado final
df_atribucion.head()

Total de Ventas: 12
Total de venta atribuida: 2,274,330


,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,contactid,channel
0,C1037574868,NATALIA RICO GIL,NATYRICOG@HOTMAIL.COM,3008243240,2025-07-19,Chat Center,161344.53,3008243240,whatsapp
1,C1040731931,MÓNICA CAMPIÑO,monaleja78@gmail.com,3102706431,2025-07-19,CHATCENTER WEB,106508.02,3102706431,whatsapp
2,C1048019921,ANA MARIA DURANGO VARGAS,ANA.DUVAR-1995@HOTMAIL.COM,3006595205,2025-07-19,Chat Center,142436.97,3006595205,whatsapp
3,C1129492481,DANIELA CERRA,d_cerra16@hotmail.com,3023884448,2025-07-19,CHATCENTER WEB,106508.02,3023884448,whatsapp
4,C1140839826,LORENA BOLIVAR,LORENABOLIVAR18@GMAIL.COM,3187170156,2025-07-19,Chat Center,180252.10,3187170156,whatsapp


In [81]:
df_atribucion.to_excel('Ventas.xlsx', index=False)